In [1]:
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from dotenv import load_dotenv

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch import nn
from transformers import (
    Trainer, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
)

load_dotenv()

True

In [2]:
import ast

# Load raw TSV (no binarisation yet)
df = pd.read_csv("../../data/dontpatronizeme_pcl.tsv", sep="	", header=None, skiprows=4)
df.columns = ["par_id", "article_id", "keyword", "country_code", "text", "label_scale"]

# Load split files -- these have the 7-annotator vote lists
df_train_dev_labels = pd.read_csv("../../data/train_semeval_parids-labels.csv")
df_test_labels = pd.read_csv("../../data/dev_semeval_parids-labels.csv")


# CORRECT binarisation: any annotator voted PCL -> label=1
def to_binary(s): return 1 if sum(ast.literal_eval(s)) > 0 else 0


df_train_dev_labels["label_pcl"] = df_train_dev_labels["label"].apply(to_binary)
df_test_labels["label_pcl"] = df_test_labels["label"].apply(to_binary)

# Merge labels into main df
df = df.merge(df_train_dev_labels[["par_id", "label_pcl"]], on="par_id", how="left")
df_test_pcl = df_test_labels.merge(df[["par_id", "text", "keyword", "country_code"]], on="par_id")

print(f"Total rows with labels: {df.label_pcl.notna().sum()}")
print(f"Test rows: {len(df_test_pcl)}")
print(f"PCL% in labelled set: {df.label_pcl.mean() * 100:.1f}%")


10469

In [3]:
# Prepend keyword + country as context signal (keep stop-word removal minimal)
custom_stopwords = set()  # Removed aggressive stop-word filtering -- RoBERTa handles this


def clean_and_prepend(row):
    text = str(row["text"]).strip()
    return f"Keyword: {row["keyword"]}. Country: {row["country_code"]}. Text: {text}"


df["cleaned_text"] = df.apply(clean_and_prepend, axis=1)
df_test_pcl["cleaned_text"] = df_test_pcl.apply(clean_and_prepend, axis=1)
df.head(2)


,par_id,art_id,keyword,country_code,text,label,label_pcl,cleaned_text
0,1,@@24942188,hopeless,ph,"We 're living in times of absolute insanity , ...",0,0,Keyword: hopeless. Country: ph. Text: we 're l...
1,2,@@21968160,migrant,gh,"In Libya today , there are countless number of...",0,0,Keyword: migrant. Country: gh. Text: in libya ...
2,3,@@16584954,immigrant,ie,White House press secretary Sean Spicer said t...,0,0,Keyword: immigrant. Country: ie. Text: white h...
3,4,@@7811231,disabled,nz,Council customers only signs would be displaye...,0,0,Keyword: disabled. Country: nz. Text: council ...
4,5,@@1494111,refugee,ca,""" Just like we received migrants fleeing El Sa...",0,0,"Keyword: refugee. Country: ca. Text: "" just li..."


In [4]:
# Split files already loaded above -- just verify sizes
print(f"Train+dev labelled: {df_train_dev_labels.shape}")
print(f"Test labelled:      {df_test_labels.shape}")


(8375, 2094, 10469)

In [5]:
df_train_dev = df[df["par_id"].isin(df_train_dev_labels["par_id"])].dropna(subset=["label_pcl"])
df_train_dev["label_pcl"] = df_train_dev["label_pcl"].astype(int)
print(f"Train+dev rows: {len(df_train_dev)}")
print(df_train_dev["label_pcl"].value_counts())


8375

In [6]:
df_test = df_test_pcl.copy()
df_test["label_pcl"] = df_test["label_pcl"].astype(int)
print(f"Test rows: {len(df_test)}")


2094

In [7]:
df_train, df_dev = train_test_split(
    df_train_dev,
    test_size=0.1,
    random_state=42,
    shuffle=True,
    stratify=df_train_dev["label_pcl"],
)
print(f"Train: {len(df_train)} | Dev: {len(df_dev)}")
print(f"Train PCL%: {df_train.label_pcl.mean() * 100:.1f}%  (before augmentation)")


(6700, 1675)

In [ ]:
# ==========================================================================
# DATA AUGMENTATION FOR PCL MINORITY CLASS
# ==========================================================================
# Install once:
#   pip install sentencepiece sacremoses nlpaug anthropic
#
# AUG_METHOD options:
#   "backtranslation" - EN->pivot->EN via Helsinki-NLP MarianMT (recommended)
#                       Fully offline after first model download (~5 min to run)
#   "contextual"      - word substitution via RoBERTa MLM (fast)
#   "llm"             - paraphrase via Claude API (best quality, needs API key)
# ==========================================================================

import os

AUG_METHOD = "backtranslation"  # <-- change to switch method
N_COPIES = 2  # augmented copies per original PCL example


# (2 copies + original = 3x total, ~3:1 ratio)


# --------------------------------------------------------------------------
# Method 1: Back-translation via Helsinki-NLP MarianMT
# Translates EN -> pivot language -> EN, producing natural paraphrases that
# preserve meaning but diversify surface form. Using a different pivot language
# per copy (DE, FR, ES) maximises lexical diversity across augmented examples.
# --------------------------------------------------------------------------
def augment_backtranslation(texts, pivot_lang="de"):
    from transformers import MarianMTModel, MarianTokenizer

    def load_marian(src, tgt):
        name = f"Helsinki-NLP/opus-mt-{src}-{tgt}"
        print(f"    Loading {name}...")
        tok = MarianTokenizer.from_pretrained(name)
        mdl = MarianMTModel.from_pretrained(name)
        return tok, mdl

    def translate_batch(texts, tok, mdl, batch_size=16):
        results = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            inputs = tok(
                batch, return_tensors="pt", padding=True,
                truncation=True, max_length=512
            )
            generated = mdl.generate(**inputs, num_beams=4, max_length=512)
            results += [tok.decode(t, skip_special_tokens=True) for t in generated]
        return results

    fwd_tok, fwd_mdl = load_marian("en", pivot_lang)
    bwd_tok, bwd_mdl = load_marian(pivot_lang, "en")
    pivot_texts = translate_batch(texts, fwd_tok, fwd_mdl)
    back_texts = translate_batch(pivot_texts, bwd_tok, bwd_mdl)
    return back_texts


# --------------------------------------------------------------------------
# Method 2: Contextual word substitution via RoBERTa MLM
# Replaces ~15% of tokens with contextually plausible alternatives predicted
# by a masked language model. Fast and requires no external API, but produces
# less structural diversity than back-translation.
# --------------------------------------------------------------------------
def augment_contextual(texts, aug_p=0.15):
    import torch
    import nlpaug.augmenter.word as naw
    aug = naw.ContextualWordEmbsAug(
        model_path="roberta-base",
        action="substitute",
        aug_p=aug_p,
        device="cuda" if torch.cuda.is_available() else "cpu",
    )
    results = []
    for i in range(0, len(texts), 32):
        batch = texts[i:i + 32]
        results += aug.augment(batch)
    return results


# --------------------------------------------------------------------------
# Method 3: LLM paraphrase via Claude API
# Prompts Claude to rephrase each PCL example while explicitly preserving
# the patronising tone. Produces the highest-quality and most varied
# augmentations, but requires an ANTHROPIC_API_KEY env variable.
# --------------------------------------------------------------------------
def augment_llm(texts):
    import anthropic
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    system = (
        "You are a linguistic data augmentation tool. "
        "Paraphrase the given text, preserving any patronising, condescending, "
        "or pitying tone toward the subject. "
        "Change the wording and sentence structure but keep the same underlying "
        "meaning and attitude. Return only the paraphrased text, nothing else."
    )
    results = []
    for i, text in enumerate(texts):
        msg = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=300,
            system=system,
            messages=[{"role": "user", "content": text}]
        )
        results.append(msg.content[0].text.strip())
        if i % 50 == 0:
            print(f"    LLM augmented {i}/{len(texts)}")
    return results


# --------------------------------------------------------------------------
# Run augmentation on PCL=1 training examples only
# IMPORTANT: augment training set only -- dev/test remain untouched
# --------------------------------------------------------------------------
pcl_train = df_train[df_train["label_pcl"] == 1].copy()
pivot_langs = ["de", "fr", "es"]  # different pivot per copy for diversity
augmented_dfs = []

print(f"Augmenting {len(pcl_train)} PCL examples x{N_COPIES} copies using [{AUG_METHOD}]")

for copy_i in range(N_COPIES):
    raw_texts = pcl_train["text"].tolist()

    if AUG_METHOD == "backtranslation":
        pivot = pivot_langs[copy_i % len(pivot_langs)]
        print(f"  Copy {copy_i + 1}/{N_COPIES} | EN -> {pivot} -> EN")
        aug_texts = augment_backtranslation(raw_texts, pivot_lang=pivot)

    elif AUG_METHOD == "contextual":
        aug_p = 0.15 + copy_i * 0.05  # slightly higher replacement each copy
        print(f"  Copy {copy_i + 1}/{N_COPIES} | aug_p={aug_p}")
        aug_texts = augment_contextual(raw_texts, aug_p=aug_p)

    elif AUG_METHOD == "llm":
        print(f"  Copy {copy_i + 1}/{N_COPIES} | LLM paraphrase")
        aug_texts = augment_llm(raw_texts)

    else:
        raise ValueError(f"Unknown AUG_METHOD: {AUG_METHOD}")

    aug_df = pcl_train.copy()
    aug_df["text"] = aug_texts
    # Re-run the keyword/country prepend on the newly augmented text
    aug_df["cleaned_text"] = aug_df.apply(
        lambda r: f"Keyword: {r['keyword']}. Country: {r['country_code']}. Text: {r['text']}",
        axis=1
    )
    augmented_dfs.append(aug_df)

    print(f"    ORIGINAL:  {raw_texts[0][:120]}")
    print(f"    AUGMENTED: {aug_texts[0][:120]}")

df_train_aug = pd.concat(
    [df_train] + augmented_dfs, ignore_index=True
).sample(frac=1, random_state=42)

print(f"\nFinal train size: {len(df_train_aug):,}")
print(f"PCL% after augmentation: {df_train_aug.label_pcl.mean() * 100:.1f}%")
print(df_train_aug["label_pcl"].value_counts())


In [8]:
df_train["label_pcl"].value_counts(normalize=True)

label_pcl
0    0.905224
1    0.094776
Name: proportion, dtype: float64

In [9]:
df_dev["label_pcl"].value_counts(normalize=True)

label_pcl
0    0.905075
1    0.094925
Name: proportion, dtype: float64

In [10]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")


def tokenize_function(examples):
    return tokenizer(examples["cleaned_text"], padding="max_length", truncation=True, max_length=128)


In [11]:
hf_dataset_train = Dataset.from_pandas(df_train_aug)  # augmented training set
tokenized_dataset_train = hf_dataset_train.map(tokenize_function, batched=True)
tokenized_dataset_train = tokenized_dataset_train.rename_column("label_pcl", "labels")
tokenized_dataset_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/6700 [00:00<?, ? examples/s]

In [12]:
hf_dataset_dev = Dataset.from_pandas(df_dev)
tokenized_dataset_dev = hf_dataset_dev.map(tokenize_function, batched=True)
tokenized_dataset_dev = tokenized_dataset_dev.rename_column("label_pcl", "labels")
tokenized_dataset_dev.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/1675 [00:00<?, ? examples/s]

In [13]:
# Check for Apple Silicon (mps), then NVIDIA (cuda), then fall back to CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
device

device(type='mps')

In [14]:
class_weights = compute_class_weight(
    "balanced",
    classes=np.unique(df_train_aug["label_pcl"]),
    y=df_train_aug["label_pcl"]
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"Class weights: {class_weights}")


class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(outputs.logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


In [34]:
from sklearn.metrics import f1_score as sk_f1


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1 = sk_f1(labels, preds, pos_label=1)
    return {"f1_pcl": f1}


model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

training_args = TrainingArguments(
    output_dir="",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=8,  # was 45 -- 5-8 is correct for RoBERTa fine-tuning
    per_device_train_batch_size=16,  # was 64 -- smaller = more stable gradients
    per_device_eval_batch_size=32,
    learning_rate=2e-5,  # was default 5e-5 -- 2e-5 is standard for RoBERTa
    warmup_ratio=0.1,  # linear warmup over first 10% of steps
    weight_decay=0.01,
    load_best_model_at_end=True,  # auto-saves best checkpoint
    metric_for_best_model="f1_pcl",  # use F1 not loss for model selection
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),  # mixed precision on GPU
)

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset_train,
    eval_dataset=tokenized_dataset_dev,
    compute_metrics=compute_metrics,  # now Trainer tracks F1 every epoch
)

trainer.train()  # removed resume_from_checkpoint -- fresh run with fixed setup


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddi

Epoch,Training Loss,Validation Loss
31,0.484685,0.455017
32,0.484685,0.580680
33,0.484685,0.506148
34,0.472410,0.527846
35,0.472410,0.446997
36,0.472410,0.473749
37,0.472410,0.565832
38,0.472410,0.585915
39,0.347526,0.501526
40,0.347526,0.503428


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/henrytsyu/Imperial/70016-cw/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/henrytsyu/Imperial/70016-cw/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/henrytsyu/Imperial/70016-cw/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=4725, training_loss=0.10859435631484582, metrics={'train_runtime': 1663.866, 'train_samples_per_second': 181.204, 'train_steps_per_second': 2.84, 'total_flos': 1.190419658870784e+16, 'train_loss': 0.10859435631484582, 'epoch': 45.0})

In [35]:
from sklearn.metrics import f1_score

# 1. Get the model's predictions on your internal dev set
predictions = trainer.predict(tokenized_dataset_dev)

# 2. Convert raw logits into 0 or 1 labels
preds = np.argmax(predictions.predictions, axis=-1)

# 3. Get the true labels
labels = predictions.label_ids

# 4. Calculate the F1 score for the positive class (PCL = 1)
f1 = f1_score(labels, preds)
print("Internal Dev F1 Score:", f1)

/Users/henrytsyu/Imperial/70016-cw/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Internal Dev F1 Score: 0.5013333333333333


In [36]:
from sklearn.metrics import classification_report

# This will show you exactly how many False Positives vs False Negatives you have
print(classification_report(labels, preds))

              precision    recall  f1-score   support

           0       0.96      0.92      0.94      1516
           1       0.44      0.59      0.50       159

    accuracy                           0.89      1675
   macro avg       0.70      0.76      0.72      1675
weighted avg       0.91      0.89      0.90      1675



In [37]:
trainer.save_model("./BestModel")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [38]:
tokenizer.save_pretrained("./BestModel")

('./BestModel/tokenizer_config.json', './BestModel/tokenizer.json')

In [39]:
# Create dev.txt

hf_dataset_test = Dataset.from_pandas(df_test)
tokenized_dataset_test = hf_dataset_test.map(tokenize_function, batched=True)
tokenized_dataset_test = tokenized_dataset_test.rename_column("label_pcl", "labels")
tokenized_dataset_test.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# 1. Get the model's predictions on test set
predictions = trainer.predict(tokenized_dataset_test)

# 2. Convert raw logits into 0 or 1 labels
preds = np.argmax(predictions.predictions, axis=-1)

# 3. Get the true labels
labels = predictions.label_ids

# 4. Calculate the F1 score for the positive class (PCL = 1)
f1 = f1_score(labels, preds)
print("Test F1 Score:", f1)

Map:   0%|          | 0/2094 [00:00<?, ? examples/s]

/Users/henrytsyu/Imperial/70016-cw/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Test F1 Score: 0.47558386411889597


In [40]:
print(classification_report(labels, preds))

              precision    recall  f1-score   support

           0       0.95      0.92      0.93      1895
           1       0.41      0.56      0.48       199

    accuracy                           0.88      2094
   macro avg       0.68      0.74      0.70      2094
weighted avg       0.90      0.88      0.89      2094

